# LoRA Fine-Tuning — STAT 453, Spring 2026
**Model:** Llama 3.2 1B Instruct | **Dataset:** RECAST-30K

### What this notebook does
1. Takes the prepared RECAST dataset
2. Loads Llama 3.2 1B as a frozen base model
3. Injects small trainable LoRA adapter layers into the attention weights
4. Trains **only** those adapter layers (not the full model)
5. Saves the tiny adapter file (~30-60 MB) for evaluation

> **Runtime:** Set to **A100 GPU** (`Runtime → Change runtime type → A100`)

## Install Dependencies

In [1]:
!pip install -q transformers peft datasets accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 13.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.9 MB/s eta 0:00:00:00:0100:01


## HuggingFace Login
Required to access the gated Llama 3.2 model. Set your token in Colab Secrets (`🔑` icon in the left panel) as `HF_TOKEN`.

In [2]:
import os
from huggingface_hub import login

# Colab Secrets approach (recommended)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '').strip()

login(token=hf_token)

LocalProtocolError: Illegal header value b'Bearer '

## Imports

In [6]:
import json
import torch
from datasets import Dataset, DatasetDict
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

## Section 1: Configuration
All hyperparameters are in one place — change them here, nothing else needs to be touched.

In [7]:
# --- Model ---
# Llama 3.2 1B Instruct: already fine-tuned to follow instructions;
# we build on top of that rather than starting from scratch.
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

# --- Dataset ---
DATASET_PATH = "/content/recast_30k_clean.jsonl"  # extracted by the unzip cell above

# --- LoRA Hyperparameters ---
LORA_RANK     = 8            # higher = more expressive but more memory; sweep r=8 and r=16
LORA_ALPHA    = LORA_RANK * 2  # standard: alpha = 2 * rank
LORA_DROPOUT  = 0.05         # 5% dropout to prevent overfitting
TARGET_MODULES = ["q_proj", "v_proj"]  # query + value projections in attention

# --- Training Hyperparameters ---
LEARNING_RATE           = 1e-4  # sweep: 1e-4 (fast), 5e-5 (medium), 1e-5 (careful)
NUM_EPOCHS              = 1
PER_DEVICE_BATCH_SIZE   = 8
GRAD_ACCUMULATION_STEPS = 4    # effective batch size = 8 * 4 = 32
MAX_SEQ_LENGTH          = 512

# --- Output ---
OUTPUT_DIR = f"./outputs/lora_r{LORA_RANK}_lr{LEARNING_RATE}"

print("Configuration loaded.")
print(f"  Model:   {MODEL_NAME}")
print(f"  Rank:    {LORA_RANK}  |  Alpha: {LORA_ALPHA}  |  LR: {LEARNING_RATE}")

Configuration loaded.
  Model:   meta-llama/Llama-3.2-1B-Instruct
  Rank:    8  |  Alpha: 16  |  LR: 0.0001


## (Optional) Mount Google Drive
If your `datalora.json` lives in Drive, run this cell and update `DATASET_PATH` above.

In [8]:
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = "/content/drive/MyDrive/your-folder/datalora.json"

## Upload and Unzip Dataset
Upload `recast_30k_clean.jsonl.zip` using the Files panel (📁) or from Drive, then run this cell to extract it.

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Github repo to use in Collab

In [15]:
# Clone your GitHub repo into Colab
!git clone https://github.com/rinklerosere-cloud/stat-453-constraint-based-llm.git

!find stat-453-constraint-based-llm/ -name "*.zip"


fatal: destination path 'stat-453-constraint-based-llm' already exists and is not an empty directory.
stat-453-constraint-based-llm/datasets/recast_30k_original.jsonl.zip
stat-453-constraint-based-llm/datasets/recast_30k_clean.jsonl.zip


Finding the path of the dataset


In [16]:
# Then copy the zip file to /content
!cp stat-453-constraint-based-llm/datasets/recast_30k_clean.jsonl.zip /content/
!ls /content/recast_30k_clean.jsonl.zip

/content/recast_30k_clean.jsonl.zip


In [17]:
import zipfile, os

ZIP_PATH    = "/content/recast_30k_clean.jsonl.zip"  # update if uploading from Drive
EXTRACT_DIR = "/content"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)
    extracted = z.namelist()

print("Extracted:", extracted)

Extracted: ['recast_30k_clean.jsonl']


## Section 2: Prompt Formatting
Llama 3 uses a specific chat format with special tokens. We must match this **exactly** — a wrong format means the model can't tell where instructions end and responses begin.

In [18]:
PROMPT_TEMPLATE = (
    "<|begin_of_text|>"
    "<|start_header_id|>user<|end_header_id|>\n\n"
    "{instruction}"
    "<|eot_id|>"
    "<|start_header_id|>assistant<|end_header_id|>\n\n"
    "{response}"
    "<|eot_id|>"
)


def format_example(example: dict) -> dict:
    instruction = example.get("input", "")
    response    = example.get("output", "")
    return {
        "text": PROMPT_TEMPLATE.format(instruction=instruction, response=response)
    }

## Section 3: Load and Prepare the Dataset

In [19]:
def build_dataset() -> DatasetDict:
    print(f"\n[1/4] Loading dataset from: {DATASET_PATH}")

    clean = []
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            instruction = row.get("winner_prompt", "")
            response    = row.get("response_of_winner_prompt", "")
            if instruction and response:
                clean.append({
                    "text": PROMPT_TEMPLATE.format(
                        instruction=instruction,
                        response=response
                    )
                })

    print(f"      Raw examples loaded: {len(clean)}")

    dataset = Dataset.from_list(clean)
    dataset = dataset.shuffle(seed=42)
    split   = dataset.train_test_split(test_size=0.1, seed=42)

    print(f"      Train: {len(split['train'])} | Val: {len(split['test'])}")
    return DatasetDict({"train": split["train"], "validation": split["test"]})

## Section 4: Load Model and Tokenizer

In [20]:
def load_model_and_tokenizer():
    print(f"\n[2/4] Loading model: {MODEL_NAME}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    # Llama 3 has no dedicated pad token — reuse EOS token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,  # half-precision; A100 handles this natively
        device_map="auto",
    )
    model.config.use_cache = False       # must be off during training
    model.enable_input_require_grads()   # required for PEFT + gradient checkpointing

    if torch.cuda.is_available():
        mem_gb = torch.cuda.memory_reserved() / 1e9
        print(f"      Model loaded. GPU memory reserved: {mem_gb:.1f} GB")

    return model, tokenizer

## Section 5: Inject LoRA Adapters
Freezes all original model weights and injects small trainable A/B matrices.
With r=8 on `q_proj` + `v_proj` we train ~0.5% of the 1.24B parameters.

In [21]:
def apply_lora(model):
    print(f"\n[3/4] Applying LoRA adapters (rank={LORA_RANK}, alpha={LORA_ALPHA})")

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        inference_mode=False,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()  # should show ~0.5%
    return model

## Section 6: Train
**What to watch:**
- Train loss should decrease steadily
- Val loss should also decrease (more slowly)
- Val loss rising while train loss falls → overfitting → lower LR or stop early

In [22]:
def train(model, tokenizer, dataset: DatasetDict):
    print(f"\n[4/4] Starting training | LR={LEARNING_RATE} | Epochs={NUM_EPOCHS}")

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        weight_decay=0.01,
        optim="adamw_torch",
        bf16=True,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        report_to="none",  # set to "wandb" for experiment tracking
        seed=42,
    )

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
    )

    trainer.train()
    return trainer

## Section 7: Save Adapter Weights
Saves **only** the LoRA adapter (~30-60 MB), not the full 2 GB base model.

To reload later:
```python
from peft import PeftModel
base  = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
model = PeftModel.from_pretrained(base, "./outputs/.../lora_adapter")
```

In [24]:
import os

def save_adapter(trainer):
    adapter_path = os.path.join(OUTPUT_DIR, "lora_adapter")
    trainer.model.save_pretrained(adapter_path)
    trainer.processing_class.save_pretrained(adapter_path)
    print(f"\nAdapter saved to: {adapter_path}")
    print("Share this folder with Mark for the comparison runs.")
    return adapter_path

## Run the Full Pipeline

In [25]:
print("=" * 60)
print("  LoRA Fine-Tuning — STAT 453 / RECAST Project")
print(f"  Model:  {MODEL_NAME}")
print(f"  Rank:   {LORA_RANK}  |  Alpha: {LORA_ALPHA}  |  LR: {LEARNING_RATE}")
print("=" * 60)

dataset          = build_dataset()
model, tokenizer = load_model_and_tokenizer()
model            = apply_lora(model)
trainer          = train(model, tokenizer, dataset)
adapter_path     = save_adapter(trainer)

print("\nDone! Next steps:")
print(f"  1. Check val loss curve in {OUTPUT_DIR}/")
print(f"  2. Run CSR evaluation on RECAST test split using saved adapter")
print(f"  3. Share {adapter_path} with Mark")

  LoRA Fine-Tuning — STAT 453 / RECAST Project
  Model:  meta-llama/Llama-3.2-1B-Instruct
  Rank:   8  |  Alpha: 16  |  LR: 0.0001

[1/4] Loading dataset from: /content/recast_30k_clean.jsonl
      Raw examples loaded: 27349
      Train: 24614 | Val: 2735

[2/4] Loading model: meta-llama/Llama-3.2-1B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

Adding EOS to eval dataset:   0%|          | 0/2735 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2735 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss
200,2.150095,2.142741
400,2.073784,2.073174
600,2.032718,2.053326



Adapter saved to: ./outputs/lora_r8_lr0.0001/lora_adapter
Share this folder with Mark for the comparison runs.

Done! Next steps:
  1. Check val loss curve in ./outputs/lora_r8_lr0.0001/
  2. Run CSR evaluation on RECAST test split using saved adapter
  3. Share ./outputs/lora_r8_lr0.0001/lora_adapter with Mark
